In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install timm scikit-learn -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    accuracy_score, precision_score,
    recall_score, f1_score, roc_curve
)

from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
BASE_DIR = "/content/drive/MyDrive/DED_binary"
RESULT_DIR = "/content/drive/MyDrive/ded_results"

os.makedirs(RESULT_DIR, exist_ok=True)

print("Results will be saved in:", RESULT_DIR)

Results will be saved in: /content/drive/MyDrive/ded_results


In [ ]:
def collect_paths(split):
    paths = []
    labels = []
    for label in ["0", "1"]:
        folder = os.path.join(BASE_DIR, split, label)
        for img in os.listdir(folder):
            paths.append(os.path.join(folder, img))
            labels.append(int(label))
    return paths, labels

train_paths, train_labels = collect_paths("train")
val_paths, val_labels = collect_paths("valid")

all_paths = train_paths + val_paths
all_labels = train_labels + val_labels

print("Total CV Images:", len(all_paths))  # Should be 628

Total CV Images: 628


In [ ]:
class DEDDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2),
    transforms.RandomAffine(10, translate=(0.05,0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
test_paths, test_labels = collect_paths("test")
test_dataset = DEDDataset(test_paths, test_labels, transform=val_tf)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.unsqueeze(1).to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    preds = []
    true = []
    total_loss = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.unsqueeze(1).to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds.extend(torch.sigmoid(outputs).cpu().numpy())
            true.extend(labels.cpu().numpy())

    preds = np.array(preds).ravel()
    true = np.array(true).ravel()

    auc = roc_auc_score(true, preds)
    return total_loss/len(loader), auc, true, preds

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_test_preds = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
    print(f"\n===== FOLD {fold+1} =====")

    train_dataset = DEDDataset(
        [all_paths[i] for i in train_idx],
        [all_labels[i] for i in train_idx],
        transform=train_tf
    )

    val_dataset = DEDDataset(
        [all_paths[i] for i in val_idx],
        [all_labels[i] for i in val_idx],
        transform=val_tf
    )

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)
    model.to(device)

    pos_weight = torch.tensor(
        (len(train_idx) - sum([all_labels[i] for i in train_idx])) /
        sum([all_labels[i] for i in train_idx])
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    scaler = GradScaler()

    best_auc = 0
    patience = 4
    counter = 0

    for epoch in range(20):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_auc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        print(f"Epoch {epoch+1} | Val AUC: {val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), f"{RESULT_DIR}/fold_{fold+1}.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping")
                break

    fold_aucs.append(best_auc)

    # Load best model
    model.load_state_dict(torch.load(f"{RESULT_DIR}/fold_{fold+1}.pth"))

    _, _, true, preds = evaluate(model, test_loader, criterion)
    fold_test_preds.append(preds)


===== FOLD 1 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/tmp/ipython-input-3949853400.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-2051715567.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 | Val AUC: 0.9268


...

Epoch 6 | Val AUC: 0.9659


/tmp/ipython-input-2051715567.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 7 | Val AUC: 0.9523


/tmp/ipython-input-2051715567.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 8 | Val AUC: 0.9565
Early stopping


In [ ]:
ensemble_preds = np.mean(fold_test_preds, axis=0)

fpr, tpr, thresholds = roc_curve(true, ensemble_preds)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

final_preds = (ensemble_preds >= optimal_threshold).astype(int)

acc = accuracy_score(true, final_preds)
prec = precision_score(true, final_preds)
rec = recall_score(true, final_preds)
f1 = f1_score(true, final_preds)
cm = confusion_matrix(true, final_preds)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

final_auc = roc_auc_score(true, ensemble_preds)

print("\n===== FINAL ENSEMBLE RESULTS =====")
print("Accuracy:", acc)
print("Sensitivity:", rec)
print("Specificity:", specificity)
print("Precision:", prec)
print("F1:", f1)
print("AUC:", final_auc)
print("Fold AUC Mean ± Std:", np.mean(fold_aucs), "±", np.std(fold_aucs))
print("Confusion Matrix:\n", cm)

# Save fold AUCs
pd.DataFrame({
    "Fold":[1,2,3,4,5],
    "AUC":fold_aucs
}).to_csv(f"{RESULT_DIR}/fold_auc_scores.csv", index=False)

# Save final metrics
pd.DataFrame({
    "Accuracy":[acc],
    "Sensitivity":[rec],
    "Specificity":[specificity],
    "Precision":[prec],
    "F1":[f1],
    "AUC":[final_auc]
}).to_csv(f"{RESULT_DIR}/DED_5fold_results.csv", index=False)

print("All results saved successfully.")


===== FINAL ENSEMBLE RESULTS =====
Accuracy: 0.927710843373494
Sensitivity: 0.8823529411764706
Specificity: 0.9591836734693877
Precision: 0.9375
F1: 0.9090909090909091
AUC: 0.9285714285714285
Fold AUC Mean ± Std: 0.9754282765737875 ± 0.004036654340766486
Confusion Matrix:
 [[47  2]
 [ 4 30]]
All results saved successfully.
